# Labels und Selectors

Ein Service spricht zunächst Version 1.0 eines Webservers an. Durch Änderung des Selectors wird auf Version 2.0 umgeschaltet. Beide Pods bleiben unverändert aktiv.

Das Notebook aus dem Verzeichnis selectors ausführen. Die Ressourcennamen webserver, webserver-v1 und webserver-v2 müssen im aktuellen Namespace frei sein. Der NodePort 30090 muss clusterweit frei sein.

## 1. Zwei Pods bereitstellen
Beide Pods haben das Label app: webserver, aber unterschiedliche version-Labels.

In [ ]:
%%bash
set -e
cat webserver-v1.yml webserver-v2.yml
kubectl apply -f webserver-v1.yml -f webserver-v2.yml
kubectl wait --for=condition=Ready pod/webserver-v1 pod/webserver-v2 --timeout=180s
kubectl get pods -l app=webserver --show-labels

## 2. Service bereitstellen
Der Selector des Service enthält app: webserver und version: v1. Beide Labels müssen übereinstimmen.

In [ ]:
%%bash
set -e
cat service.yml
kubectl apply -f service.yml
kubectl get service webserver

## 3. Version 1.0 aufrufen
Eine erreichbare interne Node-IP aus der folgenden Ausgabe auswählen. Im Browser http://NODE_IP:30090 öffnen (NODE_IP ersetzen). Erwartet: Welcome - Version 1.0.

In [ ]:
%%bash
set -e
kubectl get nodes -o wide

In [ ]:
%%bash
set -e
NODE_IP=$(kubectl get nodes -o jsonpath='{.items[0].status.addresses[?(@.type=="InternalIP")].address}')
curl -fsS --connect-timeout 5 --max-time 10 "http://$NODE_IP:30090/"

## 4. Selector ändern
Nur das Versionskriterium des Service wird geändert. Alternativ kann der Selector interaktiv im Terminal mit kubectl edit service webserver bearbeitet werden.

In [ ]:
%%bash
set -e
kubectl patch service webserver -p '{"spec":{"selector":{"version":"v2"}}}'
kubectl get service webserver -o yaml
kubectl get pods -l app=webserver --show-labels

## 5. Version 2.0 aufrufen
Dieselbe URL erneut aufrufen (gegebenenfalls Browser aktualisieren). Nach kurzer Aktualisierungszeit wird Version 2.0 angezeigt. Beide Pods laufen weiterhin.

In [ ]:
%%bash
set -e
NODE_IP=$(kubectl get nodes -o jsonpath='{.items[0].status.addresses[?(@.type=="InternalIP")].address}')
curl -fsS --connect-timeout 5 --max-time 10 "http://$NODE_IP:30090/"

## 6. Aufräumen
Nur die für diese Übung angelegten Ressourcen löschen.

In [ ]:
%%bash
set -e
kubectl delete -f service.yml -f webserver-v1.yml -f webserver-v2.yml

## Fazit
Ein Service wählt Pods anhand ihrer Labels aus. Durch Änderung des Selectors lässt sich das Ziel ändern, ohne die Pods oder den Service neu zu erstellen.